# 01 · Forensic data audit

Programmatic integrity audit of the organizer file `bangladesh_dengue_large_dataset.csv`.

**Policy:** this notebook *reports* and never repairs. No row is dropped, clipped or imputed.
Outputs go to `reports/audit/`; the narrative verdict is in `docs/data_audit.md`.

Year-level descriptive counts below include 2025 (the locked test year) because an integrity
audit must cover the whole file. No modelling or feature decision is based on 2025 values.

In [1]:
import sys, json, warnings
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore", category=FutureWarning)
import numpy as np, pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
np.random.seed(2026)
from dengueshield.data import COLUMNS, EXPECTED_SHA256, RAW_PATH, STATIC, TARGET, WEATHER, sha256
OUT = ROOT / "reports" / "audit"; OUT.mkdir(parents=True, exist_ok=True)
summary, checks = {}, {}

## 1. File identity and schema

In [2]:
raw = pd.read_csv(RAW_PATH, dtype=str)   # strings first, so unparseable values are counted, not coerced silently
digest = sha256(RAW_PATH)
summary["file"] = {"path": str(RAW_PATH.relative_to(ROOT)), "sha256": digest, "sha256_matches_expected": digest == EXPECTED_SHA256,
                   "bytes": RAW_PATH.stat().st_size, "rows": len(raw), "n_columns": raw.shape[1]}
checks["checksum"] = digest == EXPECTED_SHA256
checks["column_order_matches_spec"] = list(raw.columns) == COLUMNS
summary["file"], list(raw.columns)

({'path': 'data/raw/bangladesh_dengue_large_dataset.csv',
  'sha256': 'a4088765084a0bcf37b86222e22f89d06ef92ad2ac68a40c8057a21e13703dd7',
  'sha256_matches_expected': True,
  'bytes': 2581377,
  'rows': 31320,
  'n_columns': 12},
 ['date',
  'city',
  'ward_id',
  'population_density_sqkm',
  'water_logging_risk_index',
  'sanitation_score',
  'avg_temp_c',
  'max_temp_c',
  'min_temp_c',
  'rainfall_mm',
  'relative_humidity_pct',
  'dengue_cases'])

## 2. Parsing, missingness, duplicates

In [3]:
df = raw.copy()
df["date"] = pd.to_datetime(raw["date"], format="%Y-%m-%d", errors="coerce")
num = [c for c in COLUMNS if c not in ("date", "city", "ward_id")]
for c in num:
    df[c] = pd.to_numeric(raw[c], errors="coerce")
parse_fail = {c: int(df[c].isna().sum() - raw[c].isna().sum()) for c in ["date", *num]}
missing = {c: int(raw[c].isna().sum() + (raw[c].str.strip() == "").sum()) for c in COLUMNS}
summary.update(parse_failures=parse_fail, missing_values=missing,
               duplicate_rows=int(raw.duplicated().sum()),
               duplicate_ward_date=int(df.duplicated(["ward_id", "date"]).sum()),
               cases_non_integer=int((df[TARGET] % 1 != 0).sum()), cases_negative=int((df[TARGET] < 0).sum()))
checks.update(no_parse_failures=not any(parse_fail.values()), no_missing=not any(missing.values()),
              no_duplicate_ward_date=summary["duplicate_ward_date"] == 0,
              cases_nonnegative_integer=summary["cases_negative"] == 0 and summary["cases_non_integer"] == 0)
pd.DataFrame({"parse_failures": parse_fail, "missing": missing}).fillna(0).astype(int).T

,date,population_density_sqkm,water_logging_risk_index,sanitation_score,avg_temp_c,max_temp_c,min_temp_c,rainfall_mm,relative_humidity_pct,dengue_cases,city,ward_id
parse_failures,0,0,0,0,0,0,0,0,0,0,0,0
missing,0,0,0,0,0,0,0,0,0,0,0,0


## 3. Calendar coverage and per-ward timestamp regularity

In [4]:
dates = df["date"].drop_duplicates().sort_values()
full = pd.date_range(dates.min(), dates.max(), freq="7D")
steps = df.sort_values(["ward_id", "date"]).groupby("ward_id")["date"].diff().dropna().dt.days
rows_per_ward = df.groupby("ward_id").size()
summary["calendar"] = {"first": str(dates.min().date()), "last": str(dates.max().date()),
                       "unique_dates": int(dates.size), "expected_weekly_dates": int(full.size),
                       "missing_weeks": [str(d.date()) for d in full.difference(dates)],
                       "weekday_counts": df["date"].dt.day_name().value_counts().to_dict(),
                       "weeks_per_year": df.drop_duplicates("date")["date"].dt.year.value_counts().sort_index().to_dict(),
                       "step_days_counts": {str(int(k)): int(v) for k, v in steps.value_counts().items()},
                       "rows_per_ward": sorted(set(rows_per_ward.tolist()))}
checks.update(no_missing_weeks=not summary["calendar"]["missing_weeks"],
              all_steps_7_days=list(summary["calendar"]["step_days_counts"]) == ["7"],
              balanced_panel=rows_per_ward.nunique() == 1)
summary["calendar"]

{'first': '2016-01-04',
 'last': '2025-12-29',
 'unique_dates': 522,
 'expected_weekly_dates': 522,
 'missing_weeks': [],
 'weekday_counts': {'Monday': 31320},
 'weeks_per_year': {2016: 52,
  2017: 52,
  2018: 53,
  2019: 52,
  2020: 52,
  2021: 52,
  2022: 52,
  2023: 52,
  2024: 53,
  2025: 52},
 'step_days_counts': {'7': 31260},
 'rows_per_ward': [522]}

## 4. Ward-to-city mapping

In [5]:
CITY_PREFIX = {"Chattogram": "CCC", "Dhaka North": "DNCC", "Dhaka South": "DSCC", "Gazipur": "GCC", "Narayanganj": "NCC"}
prefix_ok = df["ward_id"].str.split("_").str[0] == df["city"].map(CITY_PREFIX)
summary["geography"] = {"wards_per_city": df.groupby("city")["ward_id"].nunique().to_dict(),
                        "total_wards": int(df["ward_id"].nunique()),
                        "wards_in_multiple_cities": int((df.groupby("ward_id")["city"].nunique() > 1).sum()),
                        "ward_prefix_city_mismatches": int((~prefix_ok).sum())}
checks["one_city_per_ward"] = summary["geography"]["wards_in_multiple_cities"] == 0
checks["ward_prefix_matches_city"] = summary["geography"]["ward_prefix_city_mismatches"] == 0
summary["geography"]

{'wards_per_city': {'Chattogram': 10,
  'Dhaka North': 20,
  'Dhaka South': 20,
  'Gazipur': 5,
  'Narayanganj': 5},
 'total_wards': 60,
 'wards_in_multiple_cities': 0,
 'ward_prefix_city_mismatches': 0}

## 5. Static versus time-varying attributes

In [6]:
nun = df.groupby("ward_id")[STATIC + WEATHER + [TARGET]].nunique()
summary["attribute_variation"] = {c: "static_per_ward" if nun[c].max() == 1 else "time_varying" for c in nun}
static = df.groupby("ward_id")[STATIC].first().reset_index()
static.to_csv(OUT / "ward_static_attributes.csv", index=False)
print(summary["attribute_variation"])
static[["population_density_sqkm", "water_logging_risk_index", "sanitation_score"]].corr(method="spearman").round(3)

{'city': 'static_per_ward', 'population_density_sqkm': 'static_per_ward', 'water_logging_risk_index': 'static_per_ward', 'sanitation_score': 'static_per_ward', 'avg_temp_c': 'time_varying', 'max_temp_c': 'time_varying', 'min_temp_c': 'time_varying', 'rainfall_mm': 'time_varying', 'relative_humidity_pct': 'time_varying', 'dengue_cases': 'time_varying'}


,population_density_sqkm,water_logging_risk_index,sanitation_score
population_density_sqkm,1.000,0.060,-0.277
water_logging_risk_index,0.060,1.000,-0.039
sanitation_score,-0.277,-0.039,1.000


## 6. Ranges and physical plausibility

In [7]:
summary["ranges"] = df[num].describe().T[["min", "25%", "50%", "75%", "max", "mean", "std"]].round(3)
idx = df[["water_logging_risk_index", "sanitation_score"]]
summary["plausibility"] = {
    "min_temp_gt_avg": int((df.min_temp_c > df.avg_temp_c).sum()),
    "avg_temp_gt_max": int((df.avg_temp_c > df.max_temp_c).sum()),
    "rainfall_negative": int((df.rainfall_mm < 0).sum()),
    "humidity_outside_0_100": int(((df.relative_humidity_pct < 0) | (df.relative_humidity_pct > 100)).sum()),
    "humidity_exactly_99": int((df.relative_humidity_pct == 99).sum()),
    "humidity_max": float(df.relative_humidity_pct.max()),
    "weekly_rain_gt_200mm": int((df.rainfall_mm > 200).sum()),
    "avg_temp_outside_10_40": int(((df.avg_temp_c < 10) | (df.avg_temp_c > 40)).sum()),
    "index_outside_0_1": int(((idx < 0) | (idx > 1)).any(axis=1).sum()),
}
display(summary["ranges"]); summary["plausibility"]

,min,25%,50%,75%,max,mean,std
population_density_sqkm,18314.00,30573.500,42854.000,54168.250,74252.00,43157.733,14865.378
water_logging_risk_index,0.22,0.378,0.545,0.820,0.94,0.579,0.238
sanitation_score,0.25,0.380,0.570,0.712,0.88,0.555,0.192
avg_temp_c,15.74,22.880,26.850,31.022,37.51,26.923,4.482
max_temp_c,23.12,27.850,31.505,35.990,40.79,31.889,4.517
min_temp_c,13.06,18.000,21.670,25.970,31.64,21.949,4.528
rainfall_mm,0.00,0.920,8.370,59.420,268.83,35.630,50.238
relative_humidity_pct,41.84,60.468,68.220,83.980,99.00,71.176,12.969
dengue_cases,0.00,0.000,0.000,4.000,41.00,2.636,4.608


{'min_temp_gt_avg': 0,
 'avg_temp_gt_max': 0,
 'rainfall_negative': 0,
 'humidity_outside_0_100': 0,
 'humidity_exactly_99': 89,
 'humidity_max': 99.0,
 'weekly_rain_gt_200mm': 361,
 'avg_temp_outside_10_40': 0,
 'index_outside_0_1': 0}

### Robust outliers relative to the same calendar month
Flag values more than 5 robust SDs (MAD × 1.4826) from the month median. Flags are reported, not removed.

In [8]:
month = df["date"].dt.month
flags = {}
for c in WEATHER:
    med = df.groupby(month)[c].transform("median")
    mad = df.groupby(month)[c].transform(lambda x: (x - x.median()).abs().median()) * 1.4826
    flags[c] = (df[c] - med).abs() > 5 * mad.replace(0, np.nan)
flag_by_month = pd.DataFrame({c: f.groupby(month).sum() for c, f in flags.items()})
summary["weather_outliers_robust_z_gt5_within_month"] = {c: int(f.sum()) for c, f in flags.items()}
flag_by_month.to_csv(OUT / "weather_outlier_flags_by_month.csv")
flag_by_month

,avg_temp_c,max_temp_c,min_temp_c,rainfall_mm,relative_humidity_pct
date,,,,,
1,0,0,0,73,0
2,0,0,0,148,0
3,0,0,0,118,0
4,0,0,0,60,0
5,0,0,0,103,0
6,0,0,0,0,0
7,0,0,0,0,0
8,0,0,0,0,0
9,0,0,0,0,193


## 7. Synthetic-generation fingerprints

In [9]:
fp = {}
for c in ["avg_temp_c", "rainfall_mm", "relative_humidity_pct"]:
    city_mean = df.groupby(["city", "date"])[c].mean()
    nat = df.groupby("date")[c].mean()
    corr = city_mean.groupby(level="city").apply(lambda x: np.corrcoef(x.values, nat.values)[0, 1])
    fp[c] = {"mean_within_city_week_sd": round(float(df.groupby(["city", "date"])[c].std().mean()), 3),
             "sd_of_city_week_means": round(float(city_mean.std()), 3),
             "min_corr_city_vs_national_weekly_mean": round(float(corr.min()), 4)}
fp["max_minus_avg_temp"] = (df.max_temp_c - df.avg_temp_c).describe()[["mean", "std", "min", "max"]].round(3).to_dict()
fp["avg_minus_min_temp"] = (df.avg_temp_c - df.min_temp_c).describe()[["mean", "std", "min", "max"]].round(3).to_dict()
fp["corr_avg_vs_midrange_temp"] = round(float(np.corrcoef(df.avg_temp_c, (df.max_temp_c + df.min_temp_c) / 2)[0, 1]), 4)
fp["max_decimals_weather"] = int(max(raw[c].str.split(".").str[1].str.len().max() for c in WEATHER))
fp["share_rain_exactly_zero_by_month"] = (df.rainfall_mm == 0).groupby(month).mean().round(3).to_dict()
fp["monthly_mean_rainfall"] = df.groupby(month)["rainfall_mm"].mean().round(2).to_dict()
fp["zero_case_share_sd_across_wards"] = round(float(df.groupby("ward_id")[TARGET].apply(lambda x: (x == 0).mean()).std()), 4)
fp["dates_all_monday"] = bool((df["date"].dt.dayofweek == 0).all())
summary["synthetic_fingerprints"] = fp
fp

{'avg_temp_c': {'mean_within_city_week_sd': 0.581,
  'sd_of_city_week_means': 4.446,
  'min_corr_city_vs_national_weekly_mean': 0.9983},
 'rainfall_mm': {'mean_within_city_week_sd': 3.955,
  'sd_of_city_week_means': 50.086,
  'min_corr_city_vs_national_weekly_mean': 0.9993},
 'relative_humidity_pct': {'mean_within_city_week_sd': 2.41,
  'sd_of_city_week_means': 12.756,
  'min_corr_city_vs_national_weekly_mean': 0.9965},
 'max_minus_avg_temp': {'mean': 4.966, 'std': 1.044, 'min': 1.6, 'max': 8.36},
 'avg_minus_min_temp': {'mean': 4.973, 'std': 1.065, 'min': 1.6, 'max': 8.39},
 'corr_avg_vs_midrange_temp': 0.9814,
 'max_decimals_weather': 2,
 'share_rain_exactly_zero_by_month': {1: 0.393,
  2: 0.389,
  3: 0.384,
  4: 0.044,
  5: 0.06,
  6: 0.0,
  7: 0.0,
  8: 0.0,
  9: 0.026,
  10: 0.414,
  11: 0.417,
  12: 0.422},
 'monthly_mean_rainfall': {1: 2.99,
  2: 3.23,
  3: 3.53,
  4: 16.36,
  5: 24.3,
  6: 94.15,
  7: 94.21,
  8: 94.4,
  9: 85.21,
  10: 2.77,
  11: 2.89,
  12: 2.87},
 'zero_cas

## 8. Target distribution, zero inflation and dispersion

In [10]:
y = df[TARGET]
cell = df.groupby(["ward_id", month])[TARGET].agg(["mean", "var"])
cell = cell[cell["mean"] > 0.5]
summary["target"] = {"zero_share": round(float((y == 0).mean()), 4), "mean": round(float(y.mean()), 4),
                     "variance": round(float(y.var()), 4), "dispersion_index": round(float(y.var() / y.mean()), 3),
                     "median_dispersion_within_ward_month_cells": round(float((cell["var"] / cell["mean"]).median()), 3),
                     "max": int(y.max()), "total_cases": int(y.sum())}
agg = dict(total="sum", mean="mean", var="var", max="max", zero_share=lambda x: (x == 0).mean())
by_year = df.groupby(df["date"].dt.year)[TARGET].agg(**agg).round(4)
by_city = df.groupby("city")[TARGET].agg(**agg).round(4)
by_ward = df.groupby("ward_id")[TARGET].agg(**agg).round(4)
by_year.to_csv(OUT / "target_by_year.csv"); by_city.to_csv(OUT / "target_by_city.csv"); by_ward.to_csv(OUT / "target_by_ward.csv")
display(summary["target"], by_year, by_city)

{'zero_share': 0.5803,
 'mean': 2.6364,
 'variance': 21.2312,
 'dispersion_index': 8.053,
 'median_dispersion_within_ward_month_cells': 2.664,
 'max': 41,
 'total_cases': 82572}

,total,mean,var,max,zero_share
date,,,,,
2016,9151,2.9330,27.0808,41,0.5571
2017,8481,2.7183,22.8193,29,0.5878
2018,6621,2.0821,13.2896,22,0.6135
2019,9967,3.1946,31.4097,31,0.5958
2020,7669,2.4580,16.1258,29,0.5513
2021,6530,2.0929,12.5826,22,0.5776
2022,10013,3.2093,26.4509,29,0.5667
2023,9955,3.1907,31.0803,39,0.5635
2024,6760,2.1258,14.2440,29,0.6025


,total,mean,var,max,zero_share
city,,,,,
Chattogram,13470,2.5805,21.3522,38,0.5828
Dhaka North,27001,2.5863,20.7486,39,0.5846
Dhaka South,27193,2.6047,20.4688,41,0.5797
Gazipur,8295,3.1782,28.0798,34,0.5559
Narayanganj,6613,2.5337,18.8308,27,0.5854


## 9. Hard checks and saved summary

In [11]:
summary["hard_checks"] = checks
summary["hard_failures"] = [k for k, ok in checks.items() if not ok]
(OUT / "audit_summary.json").write_text(json.dumps(summary, indent=2, default=lambda o: o.to_dict() if hasattr(o, "to_dict") else str(o)))
print(json.dumps(checks, indent=2))
assert not summary["hard_failures"], summary["hard_failures"]

{
  "checksum": true,
  "column_order_matches_spec": true,
  "no_parse_failures": true,
  "no_missing": true,
  "no_duplicate_ward_date": true,
  "cases_nonnegative_integer": true,
  "no_missing_weeks": true,
  "all_steps_7_days": true,
  "balanced_panel": true,
  "one_city_per_ward": true,
  "ward_prefix_matches_city": true
}


## Verdict

All hard integrity checks pass: correct schema, no parse failures, no missing values, no duplicate
`(ward_id, date)` keys, 522 consecutive Monday-dated weeks for each of 60 wards, and one city per ward.
Physical-consistency checks also pass (min ≤ avg ≤ max temperature, non-negative rainfall, humidity ≤ 100).

The data carry clear synthetic-generation fingerprints: shared national weather with small per-ward noise,
max/min temperature built as avg ± ~5 °C, a month-step rainfall regime, and humidity capped at 99.
These are documented, not repaired. See `docs/data_audit.md` for the modelling consequences.